# Sentinel: Multi-Model Fraud Detection Comparison

This notebook trains and compares four classification models on the IEEE-CDF Fraud Detection dataset,
evaluates a rules-based engine alongside ML, and produces the visualizations used in our architecture blog post.

**Models compared:**
1. Logistic Regression (baseline)
2. Random Forest
3. XGBoost
4. LightGBM

**What we measure:**
- ROC-AUC and PR-AUC per model
- Precision/recall at various thresholds
- Rules engine performance vs ML performance vs combined

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    roc_curve, precision_recall_curve, confusion_matrix, f1_score,
    precision_score, recall_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

try:
    import mlflow
    MLFLOW_AVAILABLE = True
    mlflow.set_experiment('sentinel-model-comparison')
    print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
except Exception:
    MLFLOW_AVAILABLE = False
    print('MLflow not available - metrics will be logged locally only')

# Add project root to path for importing Sentinel modules
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

sns.set_theme(style='whitegrid', palette='colorblind')
FIGURES_DIR = Path('../docs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 2. Data Loading & Exploration

We use the IEEE-CDF Fraud Detection dataset from Kaggle (~590k transactions, ~3.5% fraud rate).
The dataset has two tables: **transactions** and **identity** (device/browser info).

If the Kaggle data isn't available, we fall back to the synthetic dataset that ships with Sentinel.

In [ ]:
KAGGLE_DIR = Path('../data/kaggle')
SYNTHETIC_PATH = Path('../data/sample_transactions.csv')

if (KAGGLE_DIR / 'train_transaction.csv').exists():
    print('Loading Kaggle IEEE-CDF dataset...')
    txn_df = pd.read_csv(KAGGLE_DIR / 'train_transaction.csv')
    id_df = pd.read_csv(KAGGLE_DIR / 'train_identity.csv')
    df = txn_df.merge(id_df, on='TransactionID', how='left')
    DATASET = 'kaggle'
    TARGET = 'isFraud'
    print(f'Loaded {len(df):,} transactions with {len(id_df):,} identity records')
else:
    print('Kaggle data not found, using synthetic dataset...')
    print('To use the full dataset, follow instructions in data/kaggle/README.md')
    df = pd.read_csv(SYNTHETIC_PATH)
    DATASET = 'synthetic'
    TARGET = 'is_fraud'
    print(f'Loaded {len(df):,} synthetic transactions')

print(f'\nFraud rate: {df[TARGET].mean():.2%}')
print(f'Shape: {df.shape}')

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[TARGET].value_counts().plot.bar(ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
axes[0].set_ylabel('Count')

# Amount distribution
amt_col = 'TransactionAmt' if DATASET == 'kaggle' else 'amount'
for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    subset = df[df[TARGET] == label][amt_col]
    axes[1].hist(np.log1p(subset), bins=50, alpha=0.6, color=color,
                 label='Legit' if label == 0 else 'Fraud')
axes[1].set_title('Log(Amount) Distribution')
axes[1].set_xlabel('log(1 + amount)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

We build features that work for both the Kaggle and synthetic datasets.
For Kaggle data, we extract richer features from the available columns.
For the synthetic data, we use the same features as Sentinel's production pipeline.

In [ ]:
def engineer_features(df, dataset_type):
    """Build feature matrix from either Kaggle or synthetic data."""
    features = pd.DataFrame()
    
    if dataset_type == 'kaggle':
        # Amount features
        features['amount_log'] = np.log1p(df['TransactionAmt'])
        features['amount_cents'] = (df['TransactionAmt'] * 100 % 100).astype(int)
        
        # Time features (TransactionDT is seconds from some reference)
        features['hour'] = (df['TransactionDT'] // 3600) % 24
        features['day_of_week'] = (df['TransactionDT'] // 86400) % 7
        
        # Card features
        for col in ['card1', 'card2', 'card3', 'card5']:
            if col in df.columns:
                features[col] = df[col].fillna(-1)
        
        for col in ['card4', 'card6']:
            if col in df.columns:
                features[col] = LabelEncoder().fit_transform(df[col].fillna('unknown'))
        
        # Product code
        features['ProductCD'] = LabelEncoder().fit_transform(df['ProductCD'].fillna('unknown'))
        
        # Address features
        features['addr1'] = df['addr1'].fillna(-1)
        features['addr2'] = df['addr2'].fillna(-1)
        
        # Email domain features
        for col in ['P_emaildomain', 'R_emaildomain']:
            if col in df.columns:
                freq = df[col].value_counts(normalize=True)
                features[f'{col}_freq'] = df[col].map(freq).fillna(0)
        
        # Count-based V features (use a subset of the most informative)
        v_cols = [f'V{i}' for i in [1, 2, 3, 12, 13, 14, 54, 75, 78, 279, 280, 294]]
        for col in v_cols:
            if col in df.columns:
                features[col] = df[col].fillna(0)
        
        # C features (counting features)
        c_cols = [f'C{i}' for i in range(1, 15)]
        for col in c_cols:
            if col in df.columns:
                features[col] = df[col].fillna(0)
        
        # D features (timedelta features)
        d_cols = [f'D{i}' for i in range(1, 16)]
        for col in d_cols:
            if col in df.columns:
                features[col] = df[col].fillna(-1)
        
        # Device info (from identity table)
        if 'DeviceType' in df.columns:
            features['DeviceType'] = LabelEncoder().fit_transform(df['DeviceType'].fillna('unknown'))
        if 'DeviceInfo' in df.columns:
            freq = df['DeviceInfo'].value_counts(normalize=True)
            features['DeviceInfo_freq'] = df['DeviceInfo'].map(freq).fillna(0)
    
    else:  # synthetic
        # Mirror Sentinel's production feature set
        features['amount_log'] = np.log1p(df['amount'])
        features['hour'] = pd.to_datetime(df['transaction_time']).dt.hour
        features['day_of_week'] = pd.to_datetime(df['transaction_time']).dt.dayofweek
        features['is_online_flag'] = df['is_online'].astype(int)
        
        cat_map = {
            'grocery': 0, 'dining': 1, 'gas': 2, 'travel': 3,
            'online_retail': 4, 'entertainment': 5, 'healthcare': 6,
            'electronics': 7, 'clothing': 8, 'services': 9,
        }
        features['merchant_cat_code'] = df['merchant_category'].map(cat_map).fillna(10)
        
        high_risk = {'NG', 'RU', 'CN', 'BR', 'IN', 'PH', 'VN'}
        features['is_high_risk_country'] = df['location_country'].isin(high_risk).astype(int)
    
    return features

X = engineer_features(df, DATASET)
y = df[TARGET].astype(int)

print(f'Feature matrix: {X.shape}')
print(f'Features: {list(X.columns)[:15]}...' if len(X.columns) > 15 else f'Features: {list(X.columns)}')

## 4. Train/Test Split & Class Imbalance Strategy

Fraud detection datasets are heavily imbalanced (~3.5% positive class). We use:
- **Stratified split** to preserve class ratios
- **`scale_pos_weight`** for tree models (tells the model to weight fraud examples higher)
- **`class_weight='balanced'`** for logistic regression (same idea)

We avoid SMOTE/oversampling here because:
1. It creates synthetic minority samples that can leak information
2. Tree-based models handle imbalance well with `scale_pos_weight`
3. We care about **precision-recall tradeoff**, not accuracy — oversampling can inflate recall at the cost of precision

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

fraud_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

print(f'Train: {len(X_train):,} ({y_train.mean():.2%} fraud)')
print(f'Test:  {len(X_test):,} ({y_test.mean():.2%} fraud)')
print(f'Scale pos weight: {fraud_ratio:.1f}')

## 5. Model Training

We train four models with increasing complexity. Each model is tracked in MLflow (if available).

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42,
            C=0.1,
        )),
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=fraud_ratio,
            eval_metric='aucpr',
            random_state=42,
        )),
    ]),
    'LightGBM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LGBMClassifier(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.1,
            scale_pos_weight=fraud_ratio,
            random_state=42,
            verbose=-1,
        )),
    ]),
}

results = {}

for name, pipeline in models.items():
    print(f'\n--- Training {name} ---')
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    metrics = {
        'auc_roc': roc_auc_score(y_test, y_prob),
        'auc_pr': average_precision_score(y_test, y_prob),
        'f1': f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
    }
    results[name] = {'metrics': metrics, 'y_prob': y_prob, 'y_pred': y_pred, 'pipeline': pipeline}
    
    print(f"  AUC-ROC: {metrics['auc_roc']:.4f}")
    print(f"  AUC-PR:  {metrics['auc_pr']:.4f}")
    print(f"  F1:      {metrics['f1']:.4f}")
    
    # Log to MLflow
    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=name.lower().replace(' ', '-')):
            clf = pipeline.named_steps['clf']
            params = clf.get_params()
            # Only log simple params
            for k, v in params.items():
                if isinstance(v, (int, float, str, bool)):
                    mlflow.log_param(k, v)
            mlflow.log_metrics(metrics)

print('\nAll models trained.')

## 6. Evaluation & Visualization

Side-by-side comparison of all four models.

In [ ]:
# Metrics comparison table
metrics_df = pd.DataFrame({name: r['metrics'] for name, r in results.items()}).T
metrics_df = metrics_df.round(4)
print('Model Comparison:')
print(metrics_df.to_string())
print(f'\nBest by AUC-PR: {metrics_df["auc_pr"].idxmax()}')

In [ ]:
# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for (name, r), color in zip(results.items(), colors):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, color=color, label=f"{name} ({r['metrics']['auc_roc']:.3f})")
    
    # PR
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    axes[1].plot(rec, prec, color=color, label=f"{name} ({r['metrics']['auc_pr']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('ROC Curves')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

axes[1].set_title('Precision-Recall Curves')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(name, fontsize=10)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Threshold Analysis

The default threshold of 0.5 isn't necessarily optimal for fraud detection.
In practice, we tune the threshold based on business requirements:
- **Lower threshold** = catch more fraud (higher recall) but more false alarms
- **Higher threshold** = fewer false alarms (higher precision) but miss more fraud

We use the best model to show how the threshold affects the precision-recall tradeoff,
and why Sentinel uses a "gray zone" for escalation.

In [ ]:
best_name = metrics_df['auc_pr'].idxmax()
best_probs = results[best_name]['y_prob']
print(f'Analyzing thresholds for best model: {best_name}')

thresholds = np.arange(0.1, 0.95, 0.05)
threshold_metrics = []

for t in thresholds:
    preds = (best_probs >= t).astype(int)
    if preds.sum() == 0:
        continue
    threshold_metrics.append({
        'threshold': t,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds, zero_division=0),
    })

tm_df = pd.DataFrame(threshold_metrics)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tm_df['threshold'], tm_df['precision'], 'b-o', markersize=4, label='Precision')
ax.plot(tm_df['threshold'], tm_df['recall'], 'r-o', markersize=4, label='Recall')
ax.plot(tm_df['threshold'], tm_df['f1'], 'g-o', markersize=4, label='F1')

# Highlight Sentinel's zones
ax.axvspan(0.0, 0.4, alpha=0.08, color='green', label='APPROVE zone')
ax.axvspan(0.4, 0.8, alpha=0.08, color='orange', label='REVIEW zone (gray)')
ax.axvspan(0.8, 1.0, alpha=0.08, color='red', label='FLAG zone')

ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Threshold Analysis — {best_name}')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Rules Engine Evaluation

Now we evaluate Sentinel's rules engine (stateless rules only) against the same test set.
This shows what a pure rules-based approach catches vs what ML catches.

In [ ]:
from sentinel.services.rules_engine import evaluate_rules_on_dataframe

# Prepare test data for rules evaluation
test_df = df.iloc[X_test.index].copy()
rules_results = evaluate_rules_on_dataframe(test_df, rules_threshold=0.5)

rules_flagged = rules_results['rules_flagged'].astype(int).values
rules_scores = rules_results['rules_score'].values

rules_precision = precision_score(y_test, rules_flagged, zero_division=0)
rules_recall = recall_score(y_test, rules_flagged)
rules_f1 = f1_score(y_test, rules_flagged, zero_division=0)

print(f'Rules Engine Performance:')
print(f'  Precision: {rules_precision:.4f}')
print(f'  Recall:    {rules_recall:.4f}')
print(f'  F1:        {rules_f1:.4f}')
print(f'  Flagged:   {rules_flagged.sum():,} / {len(rules_flagged):,}')

## 9. Layer Comparison: Rules vs ML vs Combined

The key insight of Sentinel's architecture: **combining rules + ML outperforms either alone.**

We evaluate four strategies:
- **Rules only** — flag if rules_score >= 0.5
- **ML only** — flag if ML score >= 0.5 (best model)
- **Combined OR** — flag if either triggers (max recall)
- **Combined AND** — flag if both trigger (max precision)

In [ ]:
ml_flagged = (best_probs >= 0.5).astype(int)
combined_or = ((ml_flagged == 1) | (rules_flagged == 1)).astype(int)
combined_and = ((ml_flagged == 1) & (rules_flagged == 1)).astype(int)

layer_comparison = {}
for name, preds in [('Rules Only', rules_flagged), ('ML Only', ml_flagged),
                      ('Combined OR', combined_or), ('Combined AND', combined_and)]:
    layer_comparison[name] = {
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds, zero_division=0),
    }

layer_df = pd.DataFrame(layer_comparison).T
print('Layer Comparison:')
print(layer_df.round(4).to_string())

In [ ]:
# Grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(layer_df))
width = 0.25

bars1 = ax.bar(x - width, layer_df['precision'], width, label='Precision', color='#3498db')
bars2 = ax.bar(x, layer_df['recall'], width, label='Recall', color='#e74c3c')
bars3 = ax.bar(x + width, layer_df['f1'], width, label='F1', color='#2ecc71')

ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 by Detection Layer')
ax.set_xticks(x)
ax.set_xticklabels(layer_df.index, rotation=15)
ax.legend()
ax.set_ylim(0, 1.05)

# Value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'layer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export Best Model

Save the best-performing model for use in Sentinel's production pipeline.

In [ ]:
best_pipeline = results[best_name]['pipeline']
output_path = Path('../models/fraud_model.joblib')
output_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, output_path)
print(f'Saved {best_name} pipeline to {output_path}')
print(f'Model size: {output_path.stat().st_size / 1024:.1f} KB')

if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name=f'best-model-{best_name.lower().replace(" ", "-")}'):
        mlflow.log_metrics(results[best_name]['metrics'])
        mlflow.log_artifact(str(output_path))
    print('Best model logged to MLflow')

## 10b. Export All Models to BentoML Store

BentoML's model store gives us versioned model management — we can load multiple models
simultaneously, hot-swap the champion, and remove underperforming models without restarting the service.

In [ ]:
try:
    import bentoml
    import bentoml.sklearn

    model_names = {
        'Logistic Regression': 'logreg',
        'Random Forest': 'random-forest',
        'XGBoost': 'xgboost',
        'LightGBM': 'lightgbm',
    }

    for display_name, short_name in model_names.items():
        r = results[display_name]
        tag = bentoml.sklearn.save_model(
            "sentinel-fraud",
            r['pipeline'],
            labels={"framework": short_name, "name": short_name, "dataset": DATASET},
            metadata={
                "auc_pr": float(r['metrics']['auc_pr']),
                "auc_roc": float(r['metrics']['auc_roc']),
                "f1": float(r['metrics']['f1']),
                "precision": float(r['metrics']['precision']),
                "recall": float(r['metrics']['recall']),
            },
        )
        print(f"Saved {display_name} -> {tag}")

    best_short = model_names[best_name]
    print(f"\nRecommended champion: {best_name} ({best_short})")

    # List all saved models
    print("\nModels in BentoML store:")
    print(f"{'Tag':<45} {'Framework':<15} {'AUC-PR':<10}")
    print("-" * 70)
    for model in bentoml.models.list("sentinel-fraud"):
        fw = model.info.labels.get('framework', '?')
        auc_pr = model.info.metadata.get('auc_pr', 'N/A')
        if isinstance(auc_pr, float):
            auc_pr = f"{auc_pr:.4f}"
        print(f"{str(model.tag):<45} {fw:<15} {auc_pr:<10}")

except ImportError:
    print("BentoML not installed — skipping model store export")
    print("Install with: pip install bentoml>=1.2")

## 11. Conclusions & Tradeoffs

### Key Takeaways

1. **Gradient boosting models (XGBoost, LightGBM) dominate** on structured tabular fraud data.
   Logistic Regression provides a useful baseline but lacks the capacity to capture non-linear patterns.

2. **AUC-PR is the right metric**, not AUC-ROC. With 3.5% fraud, AUC-ROC can look high even
   for a mediocre model. PR-AUC directly measures what we care about: precision and recall on the minority class.

3. **Rules and ML are complementary**:
   - Rules catch obvious patterns (high amounts, risky countries) with high precision
   - ML catches subtle patterns across many features
   - Combined OR maximizes recall; Combined AND maximizes precision

4. **The gray zone matters**: Not every transaction needs an instant yes/no.
   The review queue lets human analysts focus on cases where the system is genuinely uncertain.

5. **Threshold is a business decision**: The "right" threshold depends on the cost of missing fraud
   vs the cost of false alarms. There is no universal answer.

### Model Tradeoffs

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| Logistic Regression | Fast, interpretable, good baseline | Can't capture non-linear patterns |
| Random Forest | Handles non-linearity, robust | Slower inference, less tunable |
| XGBoost | Best balance of performance and speed | Requires careful hyperparameter tuning |
| LightGBM | Fastest training, competitive accuracy | Can overfit on small datasets |